In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance,PayloadSchemaType,PointStruct,SparseVectorParams,Document,Prefetch,FusionQuery
from qdrant_client import models

import pandas as pd
from groq import Groq
import cohere


c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
qdrant_client=QdrantClient(url="http://localhost:6333")

c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [3]:
import json
import os
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

HF_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
HF_API_TOKEN = os.environ.get('HF_API_TOKEN')  # set your Hugging Face token, e.g., os.environ.get('HF_API_TOKEN')


def _mean_pool_embedding(raw_embedding):
    if not raw_embedding:
        raise ValueError("Hugging Face API returned an empty embedding.")

    if isinstance(raw_embedding[0], (int, float)):
        return [float(value) for value in raw_embedding]

    if isinstance(raw_embedding[0], list):
        token_count = len(raw_embedding)
        vector_size = len(raw_embedding[0])
        pooled = [0.0] * vector_size

        for token_vector in raw_embedding:
            if len(token_vector) != vector_size:
                raise ValueError("Inconsistent token vector dimensions in HF embedding response.")
            for idx, value in enumerate(token_vector):
                pooled[idx] += float(value)

        return [value / token_count for value in pooled]

    raise ValueError("Unexpected Hugging Face embedding response format.")


def get_embedding(text, model_name: str | None = None):
    selected_model = model_name or HF_EMBEDDING_MODEL
    endpoint = (
        "https://router.huggingface.co/hf-inference/models/"
        f"{selected_model}/pipeline/feature-extraction"
    )
    payload = json.dumps({"inputs": text, "normalize": True}).encode("utf-8")

    headers = {"Content-Type": "application/json"}
    if HF_API_TOKEN:
        headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

    request = Request(endpoint, data=payload, headers=headers, method="POST")

    try:
        with urlopen(request, timeout=60) as response:
            response_data = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        message = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Hugging Face embedding API request failed ({exc.code}): {message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

    if isinstance(response_data, dict) and response_data.get("error"):
        raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

    if isinstance(response_data, list) and len(response_data) == 1:
        return _mean_pool_embedding(response_data[0])

    return _mean_pool_embedding(response_data)



In [4]:
def retrieve_data(query,qdrant_client, top_k=5):

    query_embedding=get_embedding(query)

    search_result=qdrant_client.query_points(
        collection_name="amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="all-MiniLM-L6-v2",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=top_k,
    )

    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]
    for search_result in search_result.points:
        retrieved_context_ids.append(search_result.payload["parent_asin"])
        retrieved_context.append(search_result.payload["description"])
        retrieved_context_ratings.append(search_result.payload["average_rating"])
        similarity_scores.append(search_result.score)


    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

In [5]:
query="can i get a rock album ?"

In [7]:
results=retrieve_data(query,qdrant_client, top_k=20)

In [8]:
results

{'retrieved_context_ids': ['B0BCXGJP5D',
  'B09P3CDB28',
  'B0B7271P1G',
  'B0BJKVLHPL',
  'B0B85VLWH4',
  'B096ZSQ21B',
  'B0B74SXZ7R',
  'B09P5RSH6P',
  'B09V6YJHC9',
  'B0B94TZTZ6',
  'B0B75289L8',
  'B09RG79R29',
  'B0BSNSTJYQ',
  'B0BM4WHYYK',
  'B09WGR2M18',
  'B09SC1955G',
  'B09QNNDJP3',
  'B0BQPY5W9T',
  'B0B7KKN5TN',
  'B0BKRX6SJ6'],
 'retrieved_context': ["Get Rollin' Deluxe Get Rollin’ is the 10th studio album by iconic rock band NICKELBACK and their first in five years. Nickelback is the 11th best selling musical act of all time with over 50,000,000 units sold and the 2nd best selling “foreign” group in the U.S. of the 2000s, behind only The Beatles. Amongst all these accolades, they’ve also been named Billboard’s “Top Rock Group of the Decade”, and they have received a staggering nine Grammy Award nominations, three American Music Awards, a World Music Award, a People’s Choice Award, twelve JUNO Awards, seven MuchMusic Video Awards, and have been inducted into Canada’s Wa

### Reranking


In [16]:


cohere_api_key = os.environ.get('CO_API_KEY')
cohere_client = cohere.ClientV2(cohere_api_key)

In [17]:
to_rerank = results["retrieved_context"]

In [18]:
to_rerank

["Get Rollin' Deluxe Get Rollin’ is the 10th studio album by iconic rock band NICKELBACK and their first in five years. Nickelback is the 11th best selling musical act of all time with over 50,000,000 units sold and the 2nd best selling “foreign” group in the U.S. of the 2000s, behind only The Beatles. Amongst all these accolades, they’ve also been named Billboard’s “Top Rock Group of the Decade”, and they have received a staggering nine Grammy Award nominations, three American Music Awards, a World Music Award, a People’s Choice Award, twelve JUNO Awards, seven MuchMusic Video Awards, and have been inducted into Canada’s Walk of Fame (2007). This version of Get Rollin’ is a digipak CD that features four bonus acoustic versions of songs on the album. Tracklist: 1. San Quentin 2. Skinny Little Missy 3. Those Days 4. High Time 5. Vegas Bomb 6. Tidal Wave 7. Does Heaven Even Know You’re Missing? 8. Steel Still Rusts 9. Horizon 10. Standing In The Dark 11. Just One More 12. High Time (Acou

In [22]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [23]:
response.results

[V2RerankResponseResultsItem(index=2, relevance_score=0.80843043),
 V2RerankResponseResultsItem(index=0, relevance_score=0.79919165),
 V2RerankResponseResultsItem(index=1, relevance_score=0.7928494),
 V2RerankResponseResultsItem(index=10, relevance_score=0.7889737),
 V2RerankResponseResultsItem(index=6, relevance_score=0.7770292),
 V2RerankResponseResultsItem(index=7, relevance_score=0.7715682),
 V2RerankResponseResultsItem(index=14, relevance_score=0.7715682),
 V2RerankResponseResultsItem(index=3, relevance_score=0.7517211),
 V2RerankResponseResultsItem(index=18, relevance_score=0.7517211),
 V2RerankResponseResultsItem(index=4, relevance_score=0.7428696),
 V2RerankResponseResultsItem(index=11, relevance_score=0.7383673),
 V2RerankResponseResultsItem(index=19, relevance_score=0.7307513),
 V2RerankResponseResultsItem(index=9, relevance_score=0.71351147),
 V2RerankResponseResultsItem(index=5, relevance_score=0.7086966),
 V2RerankResponseResultsItem(index=12, relevance_score=0.69562757),


In [25]:
rerank_results=[to_rerank[i.index] for i in response.results]

In [26]:
rerank_results

["Hot Rocks 1964-1971 - Exclusive Gold Hot Rocks 1964-1971 - Exclusive Limited Edition Gold Colored Vinyl 2LP Tracklist A1Time Is On My Side Heart Of Stone Play With Fire (I Can't Get No) Satisfaction As Tears Go By Get Off Of My Cloud Mother's Little Helper 19th Nervous Breakdown Paint It, Black Under My Thumb Ruby Tuesday Let's Spend The Night Together Jumpin' Jack Flash Street Fighting Man Sympathy For The Devil Honky Tonk Women Gimme Shelter Midnight Rambler (Live) You Can't Always Get What You Want Brown Sugar Wild Horses  *Nxw factory sealed vinyl LP. Small corner ding or crease that is solely cosmetic* Playing this record is recommended on a high quality player with anti-skate features. Other things you can try to help with skipping if you can’t adjust the anti-skate: Make sure your record player is perfectly flat - use a level tool to check it If you can’t get it flat simply by placement, try putting index cards beneath the feet of the turntable to adjust.",
 "Get Rollin' Delux